# 03 — IDD Data Preprocessing

Project: Privacy-Preserving Edge AI Framework for Intelligent Transportation

Total cells: 10 (1 Markdown + 9 Code)

Objectives:
1. Load the IDD dataset splits and class metadata.
2. Standardize image and annotation paths.
3. Inspect image dimensions and annotation availability.
4. Prepare clean dataset manifests for YOLO conversion.
5. Save the manifests for subsequent training notebooks.

Note: The test split has no annotations. Its images will be retained for inference, not ground-truth evaluation.

In [1]:
# CELL 2: Imports and configuration
from pathlib import Path
import json
import pandas as pd
from PIL import Image

ROOT = Path("/workspace/IDD_Detection")
PROJECT = Path("/workspace/ITS_Project")
OUTPUT = PROJECT / "configs"

OUTPUT.mkdir(parents=True, exist_ok=True)

metadata = json.loads((OUTPUT / "idd_metadata.json").read_text())

print("Preprocessing initialized.")
print("Dataset:", ROOT)
print("Classes:", len(metadata["classes"]))

Preprocessing initialized.
Dataset: /workspace/IDD_Detection
Classes: 15


In [2]:
# CELL 3: Load dataset split manifests
splits = {
    split: [
        line.strip()
        for line in (ROOT / f"{split}.txt").read_text().splitlines()
        if line.strip()
    ]
    for split in ["train", "val", "test"]
}

print("Train:", len(splits["train"]))
print("Validation:", len(splits["val"]))
print("Test:", len(splits["test"]))

Train: 31569
Validation: 10225
Test: 4794


In [3]:
# CELL 4: Build standardized dataset records
records = []

for split, entries in splits.items():
    for entry in entries:
        image = (ROOT / "JPEGImages" / entry).with_suffix(".jpg")
        annotation = (ROOT / "Annotations" / entry).with_suffix(".xml")

        records.append({
            "split": split,
            "entry": entry,
            "image_path": str(image),
            "annotation_path": str(annotation) if split != "test" else "",
            "image_exists": image.is_file(),
            "annotation_exists": annotation.is_file() if split != "test" else False
        })

manifest = pd.DataFrame(records)

print("Manifest records:", len(manifest))
display(manifest.head())

Manifest records: 46588


,split,entry,image_path,annotation_path,image_exists,annotation_exists
0,train,frontFar/BLR-2018-03-22_17-39-26_2_frontFar/00...,/workspace/IDD_Detection/JPEGImages/frontFar/B...,/workspace/IDD_Detection/Annotations/frontFar/...,True,True
1,train,frontFar/BLR-2018-03-22_17-39-26_2_frontFar/00...,/workspace/IDD_Detection/JPEGImages/frontFar/B...,/workspace/IDD_Detection/Annotations/frontFar/...,True,True
2,train,frontFar/BLR-2018-03-22_17-39-26_2_frontFar/00...,/workspace/IDD_Detection/JPEGImages/frontFar/B...,/workspace/IDD_Detection/Annotations/frontFar/...,True,True
3,train,frontFar/BLR-2018-03-22_17-39-26_2_frontFar/00...,/workspace/IDD_Detection/JPEGImages/frontFar/B...,/workspace/IDD_Detection/Annotations/frontFar/...,True,True
4,train,frontFar/BLR-2018-03-22_17-39-26_2_frontFar/00...,/workspace/IDD_Detection/JPEGImages/frontFar/B...,/workspace/IDD_Detection/Annotations/frontFar/...,True,True


In [4]:
# CELL 5: Remove records with missing images
manifest = manifest[manifest["image_exists"]].copy()
manifest.reset_index(drop=True, inplace=True)

print("Records with existing images:", len(manifest))
print("\nRecords per split:")
display(manifest.groupby("split").size().rename("image_count").to_frame())

print("\nAnnotated records per split:")
display(
    manifest.groupby("split")["annotation_exists"]
    .sum()
    .rename("annotated_images")
    .to_frame()
)

Records with existing images: 46588

Records per split:


,image_count
split,
test,4794
train,31569
val,10225



Annotated records per split:


,annotated_images
split,
test,0
train,31569
val,10225


In [5]:
# CELL 6: Collect image dimensions
dimension_records = []

for path in manifest["image_path"]:
    try:
        with Image.open(path) as img:
            width, height = img.size
            dimension_records.append({
                "image_path": path,
                "width": width,
                "height": height,
                "format": img.format
            })
    except (OSError, ValueError):
        dimension_records.append({
            "image_path": path,
            "width": None,
            "height": None,
            "format": None
        })

dimensions_df = pd.DataFrame(dimension_records)
manifest = manifest.merge(dimensions_df, on="image_path", how="left")

print("Image dimensions recorded:", len(manifest))
display(manifest[["width", "height"]].value_counts().head(10).rename("image_count").to_frame())

Image dimensions recorded: 46588


,,image_count
width,height,
1920,1080,39432
1280,964,5236
1921,1080,1499
1280,720,421


In [6]:
# CELL 7: Keep readable images with valid dimensions
before = len(manifest)

manifest = manifest[
    manifest["width"].notna()
    & manifest["height"].notna()
    & (manifest["width"] > 0)
    & (manifest["height"] > 0)
].copy()

manifest.reset_index(drop=True, inplace=True)

print("Unreadable or invalid images removed:", before - len(manifest))
print("Usable images remaining:", len(manifest))

Unreadable or invalid images removed: 0
Usable images remaining: 46588


In [7]:
# CELL 8: Generate preprocessing summary
split_summary = (
    manifest.groupby("split")
    .agg(
        images=("image_path", "count"),
        annotated_images=("annotation_exists", "sum"),
        average_width=("width", "mean"),
        average_height=("height", "mean")
    )
    .round(2)
)

display(split_summary)

,images,annotated_images,average_width,average_height
split,,,,
test,4794,0,1821.24,1062.09
train,31569,31569,1850.48,1067.39
val,10225,10225,1827.01,1053.10


In [8]:
# CELL 9: Save the complete manifest and split-specific CSV files
OUTPUT = PROJECT / "configs"
OUTPUT.mkdir(parents=True, exist_ok=True)

manifest.to_csv(OUTPUT / "idd_preprocessing_manifest.csv", index=False)

for split in ["train", "val", "test"]:
    manifest[manifest["split"] == split].to_csv(
        OUTPUT / f"idd_{split}_manifest.csv",
        index=False
    )

split_summary.to_csv(OUTPUT / "idd_preprocessing_summary.csv")

print("Saved manifests to:", OUTPUT)

Saved manifests to: /workspace/ITS_Project/configs


In [9]:
# CELL 10: Preprocessing completion summary
print("DATA PREPROCESSING COMPLETE")
print("-" * 40)

for split in ["train", "val", "test"]:
    subset = manifest[manifest["split"] == split]
    print(f"{split.upper()}: {len(subset):,} usable images")

print("\nManifest:", OUTPUT / "idd_preprocessing_manifest.csv")
print("Next: Notebook 04 — XML-to-YOLO Conversion")

DATA PREPROCESSING COMPLETE
----------------------------------------
TRAIN: 31,569 usable images
VAL: 10,225 usable images
TEST: 4,794 usable images

Manifest: /workspace/ITS_Project/configs/idd_preprocessing_manifest.csv
Next: Notebook 04 — XML-to-YOLO Conversion
